# Advection on a finite interval: CuPy Leapfrog + Mur

This notebook is the CuPy counterpart of the finite-interval CPU solver:

$
U^0
\;\xrightarrow{\text{Lax--Wendroff}}\;
U^1
\;\xrightarrow{\text{Leapfrog}}\;
U^2,U^3,\ldots
$

with homogeneous inflow at $x=0$ and the Mur/outgoing closure at $x=L$.

The numerical method and the CUDA C kernels are intentionally the same as in
the PyCUDA implementation.  The difference lies in the Python GPU interface:
CuPy provides NumPy-like device arrays and uses `RawModule`/`RawKernel` to
compile and launch the custom CUDA kernels.

The implementation uses direct, coalesced global-memory accesses.  The Mur
update is fused into the same CUDA kernel that advances the interior, so no
separate boundary kernel is required.  The notebook also contains an
independent NumPy implementation and an MP4 animation comparing CuPy, CPU, and
the exact characteristic solution.


## 1. Colab setup


In [1]:
# CuPy wheel for CUDA 12.x (the usual Colab CUDA family).
!pip install -q cupy-cuda12x


## 2. Imports and CuPy CUDA interface

CuPy supplies NumPy-compatible GPU arrays together with a low-level CUDA
interface.  `RawModule` compiles the two CUDA kernels and returns `RawKernel`
objects.  NumPy is retained for the independent CPU reference and validation
diagnostics.


In [2]:
from __future__ import annotations

import argparse
from dataclasses import dataclass
from pathlib import Path

import numpy as np

try:
    import cupy as cp
except ImportError as exc:
    raise SystemExit(
        "CuPy is required. On a CUDA 12.x Colab runtime install "
        "`cupy-cuda12x` and run the notebook on an NVIDIA GPU."
    ) from exc

if not cp.cuda.is_available():
    raise SystemExit("No CUDA-capable GPU is available to CuPy.")


## 3. CUDA kernels

The CUDA C implementation is deliberately unchanged with respect to the
PyCUDA version.  Only two kernels are required.  The first constructs $U^1$
with a Lax--Wendroff step; the second advances Leapfrog.  Both kernels impose
homogeneous inflow at the left boundary.

At the right boundary, the thread that computes the last interior sample
$U_{N_x-1}^{n+1}$ immediately evaluates

$
U_{N_x}^{n+1}
=
U_{N_x-1}^{n}
+
q\left(U_{N_x}^{n}-U_{N_x-1}^{n+1}\right),
\qquad
q=\frac{1-C}{1+C}.
$

This fusion removes a separate boundary-kernel launch and does not require a
grid-wide synchronization because both values are produced by the same
thread.


In [3]:
CUDA_SOURCE = r"""
extern "C" {

/* -------------------------------------------------------------------------
   Lax--Wendroff startup with fused Mur/outgoing boundary condition.

   Each thread updates one grid point.  Interior reads are contiguous and
   therefore naturally coalesced across a warp.  The thread responsible for
   i=n-1 also writes the outflow point i=n after computing the interior value
   needed by the Mur formula.
   ------------------------------------------------------------------------- */
__global__ void lax_wendroff_mur(
    const double *__restrict__ old,
    double *__restrict__ new_field,
    const int npoints,
    const double nu,
    const double q)
{
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    const int n = npoints - 1;

    if (i >= npoints) return;

    if (i == 0) {
        new_field[0] = 0.0;  // homogeneous inflow
        return;
    }

    if (i < n) {
        const double left   = old[i - 1];
        const double center = old[i];
        const double right  = old[i + 1];

        const double value = center
                           - 0.5 * nu * (right - left)
                           + 0.5 * nu * nu
                           * (right - 2.0 * center + left);

        new_field[i] = value;

        // Fused Mur/outgoing update at x=L.
        if (i == n - 1) {
            new_field[n] = old[n - 1] + q * (old[n] - value);
        }
    }
}


/* -------------------------------------------------------------------------
   Leapfrog update with fused Mur/outgoing boundary condition.

       U_i^{n+1} = U_i^{n-1} - C (U_{i+1}^n - U_{i-1}^n).

   Three rolling device arrays represent U^{n-1}, U^n, and U^{n+1}.
   ------------------------------------------------------------------------- */
__global__ void leapfrog_mur(
    const double *__restrict__ previous,
    const double *__restrict__ current,
    double *__restrict__ next_field,
    const int npoints,
    const double nu,
    const double q)
{
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    const int n = npoints - 1;

    if (i >= npoints) return;

    if (i == 0) {
        next_field[0] = 0.0;  // homogeneous inflow
        return;
    }

    if (i < n) {
        const double value = previous[i]
                           - nu * (current[i + 1] - current[i - 1]);

        next_field[i] = value;

        // Fused Mur/outgoing update at x=L.
        if (i == n - 1) {
            next_field[n] = current[n - 1] + q * (current[n] - value);
        }
    }
}

} // extern "C"
"""


## 4. Initial data and exact characteristic reference


In [4]:
Array = np.ndarray


def compact_cosine_pulse(
    x: Array, center: float, half_width: float
) -> Array:
    """Return a compact C1 cosine bell centered at ``center``."""
    if half_width <= 0.0:
        raise ValueError("half_width must be positive")

    distance = np.abs(x - center)
    values = np.zeros_like(x, dtype=np.float64)
    inside = distance <= half_width
    values[inside] = 0.5 * (
        1.0 + np.cos(np.pi * distance[inside] / half_width)
    )
    return values


def gaussian_pulse(x: Array, center: float, sigma: float) -> Array:
    """Return a Gaussian pulse with standard deviation ``sigma``."""
    if sigma <= 0.0:
        raise ValueError("sigma must be positive")
    return np.exp(-0.5 * ((x - center) / sigma) ** 2)


def initial_data(x: Array, case: str, length: float) -> Array:
    """Construct one of the predefined initial profiles."""
    if case == "cosine":
        return compact_cosine_pulse(
            x, center=0.25 * length, half_width=0.10 * length
        )
    if case == "gaussian":
        return gaussian_pulse(
            x, center=0.25 * length, sigma=0.045 * length
        )
    raise ValueError(f"Unknown initial-data case: {case!r}")


def exact_solution(
    x: Array,
    time: float,
    *,
    velocity: float,
    length: float,
    case: str,
) -> Array:
    """Evaluate the exact characteristic solution for a>0 and zero inflow."""
    if velocity <= 0.0:
        raise ValueError("This example assumes velocity > 0")
    if time < 0.0:
        raise ValueError("time must be non-negative")

    characteristic_feet = x - velocity * time
    exact = np.zeros_like(x, dtype=np.float64)
    from_initial_line = characteristic_feet >= 0.0

    if np.any(from_initial_line):
        exact[from_initial_line] = initial_data(
            characteristic_feet[from_initial_line], case, length
        )
    return exact


## 5. CPU reference: the same Leapfrog + Mur method

The CPU routines reproduce the Leapfrog branch of the final finite-interval NumPy implementation.  They are kept independent from the GPU code so that agreement between CPU and GPU is a meaningful validation rather than a comparison of two wrappers around the same implementation.


In [5]:
def lax_wendroff_mur_cpu(old: Array, courant: float) -> Array:
    """Construct U^1 with Lax--Wendroff and the Mur/outgoing closure."""
    new = np.empty_like(old)

    # Positive velocity makes x=0 the inflow boundary.
    new[0] = 0.0

    # Centered three-point Lax--Wendroff update for the interior nodes.
    left = old[:-2]
    center = old[1:-1]
    right = old[2:]
    new[1:-1] = (
        center
        - 0.5 * courant * (right - left)
        + 0.5 * courant**2 * (right - 2.0 * center + left)
    )

    # Mur/outgoing condition at the right endpoint.
    q = (1.0 - courant) / (1.0 + courant)
    new[-1] = old[-2] + q * (old[-1] - new[-2])
    return new


def solve_cpu_reference(
    initial: Array,
    *,
    courant: float,
    nsteps: int,
) -> Array:
    """Return the NumPy Leapfrog+Mur solution with Lax--Wendroff startup."""
    if nsteps < 0:
        raise ValueError("nsteps must be non-negative")
    if nsteps == 0:
        return np.asarray(initial, dtype=np.float64).copy()

    previous = np.asarray(initial, dtype=np.float64).copy()  # U^0
    current = lax_wendroff_mur_cpu(previous, courant)        # U^1

    q = (1.0 - courant) / (1.0 + courant)

    for _ in range(1, nsteps):
        next_field = np.empty_like(current)
        next_field[0] = 0.0

        next_field[1:-1] = previous[1:-1] - courant * (
            current[2:] - current[:-2]
        )

        next_field[-1] = current[-2] + q * (
            current[-1] - next_field[-2]
        )

        # Rotate references instead of copying complete arrays.
        previous, current = current, next_field

    return current


## 6. CuPy kernel compilation and GPU solver

`cp.RawModule` compiles the complete CUDA source and exposes the two kernels.
CuPy arrays remain resident on the device throughout the time loop.  The
launch syntax differs from PyCUDA, but the launch geometry and CUDA arguments
are the same.


In [6]:
class LeapfrogMurKernels:
    """Compile the CUDA source once and expose the two required RawKernels."""

    def __init__(self) -> None:
        module = cp.RawModule(
            code=CUDA_SOURCE,
            backend="nvrtc",
        )
        self.startup = module.get_function("lax_wendroff_mur")
        self.leapfrog = module.get_function("leapfrog_mur")


def solve_gpu(
    initial: Array,
    *,
    courant: float,
    nsteps: int,
    block_size: int = 256,
    kernels: LeapfrogMurKernels | None = None,
) -> tuple[Array, float]:
    """Run Lax--Wendroff startup + Leapfrog + Mur with CuPy.

    The returned time is measured with CUDA events and includes the startup
    kernel plus all Leapfrog kernel launches.  RawModule compilation,
    allocation, the initial host-to-device copy, and the final device-to-host
    copy are deliberately outside the timed interval.
    """
    if initial.ndim != 1:
        raise ValueError("initial must be a one-dimensional array")
    if initial.size < 9:
        raise ValueError("at least 9 grid points are required")
    if not (0.0 < courant <= 1.0):
        raise ValueError("Leapfrog requires 0 < Courant number <= 1")
    if nsteps < 0:
        raise ValueError("nsteps must be non-negative")
    if block_size <= 0:
        raise ValueError("block_size must be positive")

    if nsteps == 0:
        return np.asarray(initial, dtype=np.float64).copy(), 0.0

    kernels = kernels or LeapfrogMurKernels()

    npoints_host = int(initial.size)
    npoints = np.int32(npoints_host)
    nu = np.float64(courant)
    q = np.float64((1.0 - courant) / (1.0 + courant))

    # Check the requested launch geometry against the actual GPU.
    device = cp.cuda.Device()
    max_threads = int(device.attributes["MaxThreadsPerBlock"])
    if block_size > max_threads:
        raise ValueError(
            f"block_size={block_size} exceeds the device limit {max_threads}"
        )

    grid_size = (npoints_host + block_size - 1) // block_size
    grid = (grid_size,)
    block = (block_size,)

    # CuPy arrays are device-resident. Three rolling buffers are sufficient
    # for the three-level Leapfrog scheme.
    previous = cp.asarray(initial, dtype=cp.float64)  # U^0
    current = cp.empty_like(previous)                 # U^1
    next_field = cp.empty_like(previous)              # U^{n+1}

    start = cp.cuda.Event()
    stop = cp.cuda.Event()
    start.record()

    # Second-order startup. Mur is fused in the CUDA kernel.
    kernels.startup(
        grid,
        block,
        (previous, current, npoints, nu, q),
    )

    # Leapfrog time marching. Only Python references are rotated: no complete
    # device-array copy is performed between time steps.
    for _ in range(1, nsteps):
        kernels.leapfrog(
            grid,
            block,
            (previous, current, next_field, npoints, nu, q),
        )
        previous, current, next_field = current, next_field, previous

    stop.record()
    stop.synchronize()
    gpu_seconds = 1.0e-3 * cp.cuda.get_elapsed_time(start, stop)

    return cp.asnumpy(current), gpu_seconds


## 7. Problem setup and single-run CuPy validation


In [7]:
@dataclass(frozen=True)
class RunResult:
    x: Array
    gpu: Array
    cpu: Array
    exact: Array
    dt: float
    nsteps: int
    courant: float
    gpu_seconds: float
    gpu_relative_l2: float
    cpu_relative_l2: float
    gpu_cpu_difference: float


def problem_setup(
    *,
    nx: int,
    velocity: float,
    cfl: float,
    final_time: float,
    length: float,
    case: str,
):
    """Build the same grid and effective time step used by the CPU solver."""
    if nx < 8:
        raise ValueError("nx must be at least 8")
    if velocity <= 0.0:
        raise ValueError("velocity must be positive")
    if length <= 0.0:
        raise ValueError("length must be positive")
    if final_time < 0.0:
        raise ValueError("final_time must be non-negative")
    if not (0.0 < cfl <= 1.0):
        raise ValueError("cfl must satisfy 0 < cfl <= 1")

    dx = length / nx
    x = np.linspace(0.0, length, nx + 1, dtype=np.float64)
    initial = initial_data(x, case, length)

    if final_time == 0.0:
        nsteps = 0
        dt = 0.0
        courant = cfl
    else:
        tentative_dt = cfl * dx / velocity
        nsteps = max(1, int(np.ceil(final_time / tentative_dt)))
        dt = final_time / nsteps
        courant = velocity * dt / dx

    return x, initial, dx, dt, courant, nsteps


def run_validation(
    *,
    nx: int = 4096,
    velocity: float = 1.0,
    cfl: float = 0.8,
    final_time: float = 0.75,
    length: float = 1.0,
    case: str = "cosine",
    block_size: int = 256,
) -> RunResult:
    """Run GPU and CPU solvers and compare both with the exact solution."""
    x, initial, _, dt, courant, nsteps = problem_setup(
        nx=nx,
        velocity=velocity,
        cfl=cfl,
        final_time=final_time,
        length=length,
        case=case,
    )

    kernels = LeapfrogMurKernels()
    gpu, gpu_seconds = solve_gpu(
        initial,
        courant=courant,
        nsteps=nsteps,
        block_size=block_size,
        kernels=kernels,
    )
    cpu = solve_cpu_reference(
        initial,
        courant=courant,
        nsteps=nsteps,
    )
    exact = exact_solution(
        x,
        final_time,
        velocity=velocity,
        length=length,
        case=case,
    )

    initial_norm = max(float(np.linalg.norm(initial)), 1.0e-30)
    cpu_norm = max(float(np.linalg.norm(cpu)), 1.0e-30)

    result = RunResult(
        x=x,
        gpu=gpu,
        cpu=cpu,
        exact=exact,
        dt=dt,
        nsteps=nsteps,
        courant=courant,
        gpu_seconds=gpu_seconds,
        gpu_relative_l2=float(np.linalg.norm(gpu - exact) / initial_norm),
        cpu_relative_l2=float(np.linalg.norm(cpu - exact) / initial_norm),
        gpu_cpu_difference=float(np.linalg.norm(gpu - cpu) / cpu_norm),
    )

    print("=== CuPy Leapfrog + Mur validation ===")
    print(f"grid cells                = {nx}")
    print(f"time steps                = {result.nsteps}")
    print(f"dt                        = {result.dt:.12e}")
    print(f"effective Courant number  = {result.courant:.12e}")
    print(f"GPU kernel time           = {result.gpu_seconds:.6e} s")
    print(f"GPU relative L2 error     = {result.gpu_relative_l2:.12e}")
    print(f"CPU relative L2 error     = {result.cpu_relative_l2:.12e}")
    print(f"GPU-CPU difference        = {result.gpu_cpu_difference:.12e}")

    return result


In [8]:
# Representative numerical validation.
validation = run_validation(
    nx=4096,
    cfl=0.8,
    final_time=0.75,
    case="cosine",
    block_size=256,
)


=== CuPy Leapfrog + Mur validation ===
grid cells                = 4096
time steps                = 3840
dt                        = 1.953125000000e-04
effective Courant number  = 8.000000000000e-01
GPU kernel time           = 3.594723e-02 s
GPU relative L2 error     = 9.289254441686e-05
CPU relative L2 error     = 9.289254441685e-05
GPU-CPU difference        = 4.033018540718e-15


## 8. CuPy--CPU--exact validation animation

The animation is a validation experiment rather than a performance benchmark.
Device-to-host copies are performed only at selected time levels.  For a very
fine grid, the complete solution and all error norms still use all $N_x+1$
samples, while only the curves passed to Matplotlib are spatially decimated to
keep MP4 rendering manageable.


In [9]:
def solve_cpu_with_snapshots(
    initial: Array,
    *,
    courant: float,
    nsteps: int,
    snapshot_steps: Array,
) -> Array:
    """Store selected CPU Leapfrog+Mur time levels for animation."""
    requested = {int(s) for s in snapshot_steps}
    history = {}

    previous = np.asarray(initial, dtype=np.float64).copy()
    if 0 in requested:
        history[0] = previous.copy()

    if nsteps == 0:
        return np.stack([history[0]], axis=0)

    current = lax_wendroff_mur_cpu(previous, courant)
    if 1 in requested:
        history[1] = current.copy()

    q = (1.0 - courant) / (1.0 + courant)

    for step in range(1, nsteps):
        next_field = np.empty_like(current)
        next_field[0] = 0.0
        next_field[1:-1] = previous[1:-1] - courant * (
            current[2:] - current[:-2]
        )
        next_field[-1] = current[-2] + q * (
            current[-1] - next_field[-2]
        )
        previous, current = current, next_field

        accepted_step = step + 1
        if accepted_step in requested:
            history[accepted_step] = current.copy()

    return np.stack([history[int(s)] for s in snapshot_steps], axis=0)


def solve_gpu_with_snapshots(
    initial: Array,
    *,
    courant: float,
    nsteps: int,
    snapshot_steps: Array,
    block_size: int = 256,
    kernels: LeapfrogMurKernels | None = None,
) -> Array:
    """Store selected CuPy Leapfrog+Mur time levels for animation."""
    requested = {int(s) for s in snapshot_steps}
    history = {}

    initial_host = np.asarray(initial, dtype=np.float64)
    if 0 in requested:
        history[0] = initial_host.copy()

    if nsteps == 0:
        return np.stack([history[0]], axis=0)

    kernels = kernels or LeapfrogMurKernels()

    npoints_host = int(initial_host.size)
    npoints = np.int32(npoints_host)
    nu = np.float64(courant)
    q = np.float64((1.0 - courant) / (1.0 + courant))

    grid_size = (npoints_host + block_size - 1) // block_size
    grid = (grid_size,)
    block = (block_size,)

    previous = cp.asarray(initial_host, dtype=cp.float64)
    current = cp.empty_like(previous)
    next_field = cp.empty_like(previous)

    kernels.startup(
        grid,
        block,
        (previous, current, npoints, nu, q),
    )
    if 1 in requested:
        history[1] = cp.asnumpy(current)

    for step in range(1, nsteps):
        kernels.leapfrog(
            grid,
            block,
            (previous, current, next_field, npoints, nu, q),
        )
        previous, current, next_field = current, next_field, previous

        accepted_step = step + 1
        if accepted_step in requested:
            history[accepted_step] = cp.asnumpy(current)

    return np.stack([history[int(s)] for s in snapshot_steps], axis=0)

def create_gpu_cpu_reference_animation(
    *,
    nx: int = 65536,
    velocity: float = 1.0,
    cfl: float = 0.8,
    final_time: float = 0.05,
    length: float = 1.0,
    case: str = "cosine",
    block_size: int = 256,
    max_frames: int = 120,
    max_plot_points: int = 4096,
    fps: int = 25,
    dpi: int = 120,
    output: str | Path = "leapfrog_gpu_cpu_reference_65536.mp4",
    preview: bool = True,
):
    """Create an MP4 comparison of GPU, CPU, and exact solutions."""
    if max_frames < 2:
        raise ValueError("max_frames must be at least 2")
    if max_plot_points < 2:
        raise ValueError("max_plot_points must be at least 2")

    import matplotlib.pyplot as plt
    from matplotlib.animation import FuncAnimation, FFMpegWriter

    x, initial, _, dt, courant, nsteps = problem_setup(
        nx=nx,
        velocity=velocity,
        cfl=cfl,
        final_time=final_time,
        length=length,
        case=case,
    )

    nframes = min(max_frames, nsteps + 1)
    snapshot_steps = np.unique(
        np.rint(np.linspace(0, nsteps, nframes)).astype(int)
    )
    snapshot_times = snapshot_steps * dt

    kernels = LeapfrogMurKernels()
    gpu_history = solve_gpu_with_snapshots(
        initial,
        courant=courant,
        nsteps=nsteps,
        snapshot_steps=snapshot_steps,
        block_size=block_size,
        kernels=kernels,
    )
    cpu_history = solve_cpu_with_snapshots(
        initial,
        courant=courant,
        nsteps=nsteps,
        snapshot_steps=snapshot_steps,
    )

    final_denom = max(float(np.linalg.norm(cpu_history[-1])), 1.0e-30)
    final_gpu_cpu = float(
        np.linalg.norm(gpu_history[-1] - cpu_history[-1]) / final_denom
    )
    print(
        "GPU-CPU relative difference at final time = "
        f"{final_gpu_cpu:.12e}"
    )

    # Decimate only what is sent to Matplotlib; computations remain full-grid.
    plot_stride = max(1, int(np.ceil(x.size / max_plot_points)))
    plot_indices = np.arange(0, x.size, plot_stride, dtype=int)
    if plot_indices[-1] != x.size - 1:
        plot_indices = np.append(plot_indices, x.size - 1)
    x_plot = x[plot_indices]

    ymin = min(0.0, float(np.min(gpu_history)), float(np.min(cpu_history)))
    ymax = max(1.0, float(np.max(gpu_history)), float(np.max(cpu_history)))
    pad = max(0.08 * (ymax - ymin), 0.05)

    fig, ax = plt.subplots(figsize=(8.2, 4.9))
    gpu_line, = ax.plot([], [], linewidth=2.0, label="GPU CuPy")
    cpu_line, = ax.plot([], [], linewidth=1.7, label="CPU NumPy")
    exact_line, = ax.plot([], [], "--", linewidth=1.8, label="Exact")

    time_text = ax.text(0.02, 0.96, "", transform=ax.transAxes, va="top")
    error_text = ax.text(0.02, 0.86, "", transform=ax.transAxes, va="top")

    ax.set_xlim(float(x[0]), float(x[-1]))
    ax.set_ylim(ymin - pad, ymax + pad)
    ax.set_xlabel("x")
    ax.set_ylabel("u")
    ax.set_title(
        f"Leapfrog + Mur: GPU vs CPU vs exact; Nx={nx}, C={courant:.3f}"
    )
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right")
    fig.tight_layout()

    initial_norm = max(float(np.linalg.norm(initial)), 1.0e-30)

    def initialize():
        for line in (gpu_line, cpu_line, exact_line):
            line.set_data([], [])
        time_text.set_text("")
        error_text.set_text("")
        return gpu_line, cpu_line, exact_line, time_text, error_text

    def update(frame):
        time_now = float(snapshot_times[frame])
        exact_now = exact_solution(
            x,
            time_now,
            velocity=velocity,
            length=length,
            case=case,
        )
        gpu_now = gpu_history[frame]
        cpu_now = cpu_history[frame]

        gpu_line.set_data(x_plot, gpu_now[plot_indices])
        cpu_line.set_data(x_plot, cpu_now[plot_indices])
        exact_line.set_data(x_plot, exact_now[plot_indices])

        gpu_error = float(np.linalg.norm(gpu_now - exact_now) / initial_norm)
        cpu_error = float(np.linalg.norm(cpu_now - exact_now) / initial_norm)
        gpu_cpu = float(
            np.linalg.norm(gpu_now - cpu_now)
            / max(float(np.linalg.norm(cpu_now)), 1.0e-30)
        )

        time_text.set_text(f"t = {time_now:.5f}")
        error_text.set_text(
            f"GPU error = {gpu_error:.3e}\n"
            f"CPU error = {cpu_error:.3e}\n"
            f"GPU-CPU = {gpu_cpu:.3e}"
        )
        return gpu_line, cpu_line, exact_line, time_text, error_text

    animation = FuncAnimation(
        fig,
        update,
        frames=len(snapshot_steps),
        init_func=initialize,
        interval=1000.0 / fps,
        blit=True,
    )

    output = Path(output)
    output.parent.mkdir(parents=True, exist_ok=True)

    if not FFMpegWriter.isAvailable():
        plt.close(fig)
        raise RuntimeError(
            "FFmpeg is not available. In Google Colab it is normally "
            "preinstalled; otherwise install ffmpeg before creating the MP4."
        )

    writer = FFMpegWriter(
        fps=fps,
        bitrate=2200,
        metadata={"title": "Leapfrog + Mur: CuPy, CPU, exact"},
    )
    animation.save(str(output), writer=writer, dpi=dpi)
    plt.close(fig)

    print(
        f"Video saved to {output.resolve()} "
        f"({len(snapshot_steps)} frames)."
    )

    if preview:
        from IPython.display import Video, display
        display(Video(str(output), embed=True))

    return output.resolve()


In [10]:
# High-resolution CuPy--CPU--exact animation.
video_path = create_gpu_cpu_reference_animation(
    nx=2**16,
    cfl=0.8,
    final_time=0.05,
    case="cosine",
    max_frames=120,
    max_plot_points=4096,
    output="leapfrog_gpu_cpu_reference_65536.mp4",
    preview=True,
)


GPU-CPU relative difference at final time = 2.482860183404e-15
Video saved to /content/leapfrog_gpu_cpu_reference_65536.mp4 (120 frames).


## 9. Optional command-line interface

The notebook does not call `main()` automatically, so it is safe to run from
top to bottom in Colab.  If exported to a Python script, call `main()` from the
usual `if __name__ == "__main__"` block.


In [11]:
def build_parser() -> argparse.ArgumentParser:
    """Build the command-line parser without reading process arguments."""
    parser = argparse.ArgumentParser(
        description="CuPy Leapfrog + Mur solver for 1D advection."
    )
    parser.add_argument("--case", choices=["cosine", "gaussian"], default="cosine")
    parser.add_argument("--nx", type=int, default=4096)
    parser.add_argument("--velocity", type=float, default=1.0)
    parser.add_argument("--cfl", type=float, default=0.8)
    parser.add_argument("--final-time", type=float, default=0.75)
    parser.add_argument("--length", type=float, default=1.0)
    parser.add_argument("--block-size", type=int, default=256)
    return parser


def main(argv=None) -> int:
    """Command-line entry point; pass an explicit argv list inside notebooks."""
    args = build_parser().parse_args(argv)
    run_validation(
        nx=args.nx,
        velocity=args.velocity,
        cfl=args.cfl,
        final_time=args.final_time,
        length=args.length,
        case=args.case,
        block_size=args.block_size,
    )
    return 0
